In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, RadioButtons, VBox, HBox, HTML, interactive_output, Layout
from IPython.display import display

# ============================================================
# VECTOR RANDOM PROCESSES - DUAL CHANNEL SYSTEM
# ============================================================

# ============================================================
# CSS
# ============================================================

display(HTML("""
<style>
.vector-radio .widget-radio-box {
    display: flex !important;
    flex-direction: column !important;
    gap: 3px !important;
}
.vector-radio > label {
    display: none !important;
}
</style>
"""))

# ============================================================
# SHORT DOCUMENTATION
# ============================================================

documentation = HTML("""
<div style="
    font-family:Arial, sans-serif;
    font-size:16px;
    line-height:1.40;
    width:1050px;
    margin-bottom:8px;
">

<div style="
    font-size:22px;
    font-weight:bold;
    color:#6a3d8f;
    margin-bottom:8px;
">
Vector Random Processes and a Dual-Channel LTI System
</div>

<div style="margin-bottom:4px;">
<b>Input:</b> two jointly WSS random processes form the vector x[n] = [x₁[n], x₂[n]]ᵀ.
</div>

<div style="margin-bottom:4px;">
<b>System:</b> a 2×2 frequency-response matrix H(ω) transforms the input vector into the output vector.
</div>

<div style="margin-bottom:4px;">
The output spectral-density matrix satisfies <b>Sᵧᵧ(ω) = H(ω) Sₓₓ(ω) Hᴴ(ω)</b>.
</div>

<div>
<b>This notebook:</b> compares uncoupled and coupled system channels and shows how auto- and cross-spectral quantities are transformed.
</div>

</div>
""")

# ============================================================
# RADIO BUTTONS
# ============================================================

mode_selector = RadioButtons(
    options=['Uncoupled channels', 'Coupled channels'],
    value='Uncoupled channels',
    description='',
    layout=Layout(width='190px')
)

mode_selector.add_class('vector-radio')

# ============================================================
# SLIDERS
# ============================================================

slider_style = {'description_width': '105px'}

rho_slider = FloatSlider(
    min=-0.90,
    max=0.90,
    step=0.05,
    value=0.50,
    description='Input corr. ρ:',
    continuous_update=True,
    readout=True,
    readout_format='.2f',
    style=slider_style,
    layout=Layout(width='265px')
)

sigma1_slider = FloatSlider(
    min=0.50,
    max=2.00,
    step=0.10,
    value=1.00,
    description='Std. dev. σ₁:',
    continuous_update=True,
    readout=True,
    readout_format='.1f',
    style=slider_style,
    layout=Layout(width='265px')
)

sigma2_slider = FloatSlider(
    min=0.50,
    max=2.00,
    step=0.10,
    value=1.30,
    description='Std. dev. σ₂:',
    continuous_update=True,
    readout=True,
    readout_format='.1f',
    style=slider_style,
    layout=Layout(width='265px')
)

alpha_slider = FloatSlider(
    min=0.10,
    max=0.90,
    step=0.05,
    value=0.70,
    description='LP parameter α:',
    continuous_update=True,
    readout=True,
    readout_format='.2f',
    style=slider_style,
    layout=Layout(width='265px')
)

coupling_slider = FloatSlider(
    min=0.00,
    max=0.80,
    step=0.05,
    value=0.35,
    description='Coupling c:',
    continuous_update=True,
    readout=True,
    readout_format='.2f',
    style=slider_style,
    layout=Layout(width='265px'),
    disabled=True
)

# ============================================================
# ENABLE / DISABLE COUPLING
# ============================================================

def update_coupling(change):

    coupling_slider.disabled = (mode_selector.value == 'Uncoupled channels')

mode_selector.observe(
    update_coupling,
    names='value'
)

# ============================================================
# CONTROL PANEL
# ============================================================

controls_card = VBox(
    [
        HTML("""
        <div style="
            font-family:Arial;
            font-size:18px;
            font-weight:bold;
            color:#6a3d8f;
            margin-bottom:4px;
        ">
        Dual-Channel Parameters
        </div>
        """),

        HTML(
            '<div style="font-family:Arial; font-size:14px; font-weight:bold; margin-top:3px;">Channel structure</div>'
        ),

        mode_selector,

        HTML(
            '<div style="height:4px;"></div>'
        ),

        rho_slider,
        sigma1_slider,
        sigma2_slider,
        alpha_slider,
        coupling_slider
    ],
    layout=Layout(
        width='300px',
        min_width='300px',
        padding='12px 12px',
        border='1px solid #cdbbdd',
        overflow='hidden',
        margin='8px 0px 0px 14px'
    )
)

# ============================================================
# RESULT BOX
# ============================================================

result_html = HTML()

# ============================================================
# MAIN FUNCTION
# ============================================================

def plot_vector_process(mode='Uncoupled channels', rho=0.50, sigma1=1.00, sigma2=1.30, alpha=0.70, coupling=0.35):

    # --------------------------------------------------------
    # FREQUENCY AXIS
    # --------------------------------------------------------

    omega = np.linspace(
        -np.pi,
        np.pi,
        1200
    )

    # --------------------------------------------------------
    # INPUT SPECTRAL-DENSITY MATRIX
    #
    # The two input processes may be statistically correlated.
    # --------------------------------------------------------

    S11 = sigma1**2 * np.ones_like(omega)

    S22 = sigma2**2 * np.ones_like(omega)

    S12 = rho * sigma1 * sigma2 * np.ones_like(omega, dtype=complex)

    S21 = np.conj(S12)

    # --------------------------------------------------------
    # DIRECT CHANNEL RESPONSES
    #
    # H11: first-order low-pass
    # H22: normalized first-difference high-pass
    # --------------------------------------------------------

    H11 = (1.0 - alpha) / (1.0 - alpha * np.exp(-1j * omega))

    H22 = 0.5 * (1.0 - np.exp(-1j * omega))

    # --------------------------------------------------------
    # CROSS-CHANNEL RESPONSES
    # --------------------------------------------------------

    if mode == 'Uncoupled channels':

        c = 0.0

    else:

        c = coupling

    H12 = c * np.ones_like(
        omega,
        dtype=complex
    )

    H21 = c * np.ones_like(
        omega,
        dtype=complex
    )

    # --------------------------------------------------------
    # OUTPUT SPECTRAL-DENSITY MATRIX
    #
    # Syy = H Sxx H^H
    # --------------------------------------------------------

    Sy11 = np.zeros_like(
        omega,
        dtype=float
    )

    Sy22 = np.zeros_like(
        omega,
        dtype=float
    )

    Sy12 = np.zeros_like(
        omega,
        dtype=complex
    )

    for k in range(len(omega)):

        Sxx = np.array(
            [
                [S11[k], S12[k]],
                [S21[k], S22[k]]
            ],
            dtype=complex
        )

        H = np.array(
            [
                [H11[k], H12[k]],
                [H21[k], H22[k]]
            ],
            dtype=complex
        )

        Syy = H @ Sxx @ H.conj().T

        Sy11[k] = np.real(
            Syy[0, 0]
        )

        Sy22[k] = np.real(
            Syy[1, 1]
        )

        Sy12[k] = Syy[0, 1]

    # --------------------------------------------------------
    # SPECTRAL CORRELATION COEFFICIENTS
    # --------------------------------------------------------

    Mx = np.abs(S12) / np.sqrt(
        S11 * S22
    )

    My = np.abs(Sy12) / np.sqrt(
        np.maximum(
            Sy11 * Sy22,
            1e-12
        )
    )

    # ========================================================
    # SINGLE FIGURE
    # ========================================================

    fig = plt.figure(
        figsize=(8.2, 6.6)
    )

    gs = fig.add_gridspec(
        2,
        2,
        hspace=0.47,
        wspace=0.32
    )

    ax1 = fig.add_subplot(
        gs[0, 0]
    )

    ax2 = fig.add_subplot(
        gs[0, 1]
    )

    ax3 = fig.add_subplot(
        gs[1, 0]
    )

    ax4 = fig.add_subplot(
        gs[1, 1]
    )

    # ========================================================
    # GRAPH 1:
    # INPUT AUTO-SPECTRA
    # ========================================================

    ax1.plot(
        omega,
        S11,
        linewidth=2.0,
        label='Sx1x1'
    )

    ax1.plot(
        omega,
        S22,
        linewidth=2.0,
        label='Sx2x2'
    )

    ax1.set_xlim(
        -np.pi,
        np.pi
    )

    ax1.set_ylim(
        0,
        4.5
    )

    ax1.set_xticks(
        [
            -np.pi,
            0,
            np.pi
        ]
    )

    ax1.set_xticklabels(
        [
            '-π',
            '0',
            'π'
        ]
    )

    ax1.set_xlabel(
        'Angular frequency ω',
        fontsize=10
    )

    ax1.set_ylabel(
        'PSD',
        fontsize=10
    )

    ax1.set_title(
        'Input Auto-Spectra',
        fontsize=12,
        pad=8
    )

    ax1.tick_params(
        axis='both',
        labelsize=9
    )

    ax1.grid(
        True,
        linestyle=':',
        alpha=0.5
    )

    ax1.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, -0.20),
        ncol=2,
        fontsize=8
    )

    # ========================================================
    # GRAPH 2:
    # INPUT CROSS-SPECTRUM
    # ========================================================

    ax2.plot(
        omega,
        np.real(S12),
        linewidth=2.0
    )

    ax2.axhline(
        0,
        linewidth=0.8
    )

    ax2.set_xlim(
        -np.pi,
        np.pi
    )

    ax2.set_ylim(
        -4.0,
        4.0
    )

    ax2.set_xticks(
        [
            -np.pi,
            0,
            np.pi
        ]
    )

    ax2.set_xticklabels(
        [
            '-π',
            '0',
            'π'
        ]
    )

    ax2.set_xlabel(
        'Angular frequency ω',
        fontsize=10
    )

    ax2.set_ylabel(
        'Re{Sx1x2}',
        fontsize=10
    )

    ax2.set_title(
        'Input Cross-Spectrum',
        fontsize=12,
        pad=8
    )

    ax2.tick_params(
        axis='both',
        labelsize=9
    )

    ax2.grid(
        True,
        linestyle=':',
        alpha=0.5
    )

    # ========================================================
    # GRAPH 3:
    # OUTPUT AUTO-SPECTRA
    # ========================================================

    ax3.plot(
        omega,
        Sy11,
        linewidth=2.0,
        label='Sy1y1'
    )

    ax3.plot(
        omega,
        Sy22,
        linewidth=2.0,
        label='Sy2y2'
    )

    ax3.set_xlim(
        -np.pi,
        np.pi
    )

    ax3.set_ylim(
        0,
        16.0
    )

    ax3.set_xticks(
        [
            -np.pi,
            0,
            np.pi
        ]
    )

    ax3.set_xticklabels(
        [
            '-π',
            '0',
            'π'
        ]
    )

    ax3.set_xlabel(
        'Angular frequency ω',
        fontsize=10
    )

    ax3.set_ylabel(
        'PSD',
        fontsize=10
    )

    ax3.set_title(
        'Output Auto-Spectra',
        fontsize=12,
        pad=8
    )

    ax3.tick_params(
        axis='both',
        labelsize=9
    )

    ax3.grid(
        True,
        linestyle=':',
        alpha=0.5
    )

    ax3.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, -0.20),
        ncol=2,
        fontsize=8
    )

    # ========================================================
    # GRAPH 4:
    # SPECTRAL CORRELATION
    # ========================================================

    ax4.plot(
        omega,
        Mx,
        linewidth=2.0,
        label='Input |Mx1x2|'
    )

    ax4.plot(
        omega,
        My,
        linewidth=2.0,
        label='Output |My1y2|'
    )

    ax4.set_xlim(
        -np.pi,
        np.pi
    )

    ax4.set_ylim(
        0,
        1.05
    )

    ax4.set_xticks(
        [
            -np.pi,
            0,
            np.pi
        ]
    )

    ax4.set_xticklabels(
        [
            '-π',
            '0',
            'π'
        ]
    )

    ax4.set_xlabel(
        'Angular frequency ω',
        fontsize=10
    )

    ax4.set_ylabel(
        'Spectral correlation',
        fontsize=10
    )

    ax4.set_title(
        'Spectral Correlation Coefficient',
        fontsize=12,
        pad=8
    )

    ax4.tick_params(
        axis='both',
        labelsize=9
    )

    ax4.grid(
        True,
        linestyle=':',
        alpha=0.5
    )

    ax4.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, -0.20),
        ncol=1,
        fontsize=8
    )

    # ========================================================
    # FIGURE SPACING
    # ========================================================

    fig.subplots_adjust(
        left=0.09,
        right=0.97,
        top=0.93,
        bottom=0.15
    )

    plt.show()

    plt.close(fig)

    # ========================================================
    # NUMERICAL INFORMATION
    # ========================================================

    if mode == 'Uncoupled channels':

        mode_text = 'H12 = H21 = 0: there is no cross-channel interaction.'

    else:

        mode_text = f'H12 = H21 = {coupling:.2f}: cross-channel coupling is active.'

    result_html.value = f"""
    <div style="
        font-family:Arial, sans-serif;
        font-size:15px;
        line-height:1.42;
        width:760px;
        padding:10px 14px;
        border:1px solid #d7c38d;
        background:#fffbed;
        box-sizing:border-box;
    ">

    <b>Current mode:</b>
    {mode_text}

    <br>

    <b>Input spectral correlation:</b>
    |Mₓ₁ₓ₂| = {abs(rho):.3f}

    &nbsp;&nbsp;&nbsp;

    <b>Input powers:</b>
    σ₁² = {sigma1**2:.3f},
    σ₂² = {sigma2**2:.3f}

    </div>
    """

# ============================================================
# INTERACTIVE OUTPUT
# ============================================================

output = interactive_output(
    plot_vector_process,
    {
        'mode': mode_selector,
        'rho': rho_slider,
        'sigma1': sigma1_slider,
        'sigma2': sigma2_slider,
        'alpha': alpha_slider,
        'coupling': coupling_slider
    }
)

# ============================================================
# GRAPH AREA
# ============================================================

graph_area = VBox(
    [
        output,
        result_html
    ],
    layout=Layout(
        width='800px',
        min_width='800px',
        overflow='hidden'
    )
)

# ============================================================
# MAIN BODY:
# GRAPHS LEFT - CONTROLS RIGHT
# ============================================================

body_layout = HBox(
    [
        graph_area,
        controls_card
    ],
    layout=Layout(
        width='1120px',
        align_items='flex-start',
        justify_content='flex-start',
        overflow='hidden'
    )
)

# ============================================================
# SHORT INTERPRETATION
# ============================================================

interpretation = HTML("""
<div style="
    font-family:Arial, sans-serif;
    font-size:15px;
    line-height:1.42;
    width:1100px;
    padding:11px 15px;
    border:1px solid #d8cbe3;
    background:#fcf9ff;
    box-sizing:border-box;
    margin-top:6px;
">

<div style="
    font-size:18px;
    font-weight:bold;
    color:#6a3d8f;
    margin-bottom:6px;
">
Interpretation of the Results
</div>

<div style="margin-bottom:4px;">
The two input processes may already be statistically correlated, as controlled by ρ.
</div>

<div style="margin-bottom:4px;">
With uncoupled system channels, H₁₂ = H₂₁ = 0, so each output is produced only by its corresponding input channel.
</div>

<div>
With coupled channels, each output receives contributions from both inputs, modifying both the output auto-spectra and the output spectral correlation.
</div>

</div>
""")

# ============================================================
# COMPLETE NOTEBOOK
# ============================================================

main_layout = VBox(
    [
        documentation,
        body_layout,
        interpretation
    ],
    layout=Layout(
        width='1120px',
        overflow='hidden'
    )
)

display(main_layout)